<a href="https://colab.research.google.com/github/rifaf268-droid/cinebot/blob/main/cine_bot_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1: Install Dependencies
!pip install -q anthropic requests

import importlib, sys
for pkg in ["anthropic", "requests"]:
    if importlib.util.find_spec(pkg) is None:
        raise ImportError(f"Failed to install {pkg}")

print("✅ All packages ready!")
print("📌 Next: Set your API keys in Cell 2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 20.9 MB/s eta 0:00:00
✅ All packages ready!
📌 Next: Set your API keys in Cell 2


In [17]:
import os, json, requests, anthropic
from IPython.display import HTML, display, clear_output

# ─────────────────────────────────────────────
# 🔑  PASTE YOUR KEYS HERE
# ─────────────────────────────────────────────
# Initialize keys to empty strings to avoid NameError
TMDB_API_KEY = "TMDB_API_KEY_NAME"
OPENROUTER_API_KEY = "OPENROUTER_API_KEY_NAME"

# Option 1: Paste your keys directly here (uncomment and replace "YOUR_KEY_HERE")
# TMDB_API_KEY       = "YOUR_TMDB_API_KEY_HERE"
# OPENROUTER_API_KEY = "YOUR_OPENROUTER_API_KEY_HERE"


# Option 2: Load from Colab Secrets (recommended for security)
try:
    from google.colab import userdata
    # Replace "TMDB_API_KEY_NAME" and "OPENROUTER_API_KEY_NAME" with the actual names
    # you used when saving your keys in Colab Secrets.
    TMDB_API_KEY       = userdata.get("TMDB_API_KEY_NAME", "") or TMDB_API_KEY
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY_NAME", "") or OPENROUTER_API_KEY
except Exception:
    pass
# ─────────────────────────────────────────────

TMDB_BASE  = "https://api.themoviedb.org/3"
IMG_W500   = "https://image.tmdb.org/t/p/w500"
IMG_W200   = "https://image.tmdb.org/t/p/w200"
IMG_BACK   = "https://image.tmdb.org/t/p/w1280"

# TMDB genre IDs reference
GENRE_IDS = {
    "action":28, "adventure":12, "animation":16, "comedy":35,
    "crime":80,  "documentary":99, "drama":18,  "family":10751,
    "fantasy":14, "history":36,  "horror":27,  "music":10402,
    "mystery":9648, "romance":10749, "sci-fi":878, "science fiction":878,
    "thriller":53,  "war":10752,  "western":37
}

# Mood → TMDB genre IDs mapping
MOOD_GENRE_MAP = {
    "joy":         [35, 12, 10751], "happiness":   [35, 12, 10751],
    "sadness":     [18, 10749, 36], "grief":       [18, 10749],
    "anger":       [28, 12, 53],    "angry":       [28, 12, 53],
    "fear":        [35, 16, 14],    "anxiety":     [35, 16, 10751],
    "stressed":    [35, 16, 10751], "love":        [10749, 18, 10402],
    "romantic":    [10749, 18],     "surprise":    [9648, 878, 53],
    "excited":     [28, 12, 878],   "bored":       [28, 12, 878, 14],
    "nostalgic":   [18, 36, 10402], "curious":     [99, 878, 9648],
    "adventurous": [12, 28, 14, 878],"relaxed":     [35, 16, 10751],
    "neutral":     [35, 18, 12],
}

def _check_key(key, name):
    return bool(key and "YOUR_" not in key and len(key) > 10)

tmdb_ok  = _check_key(TMDB_API_KEY, "TMDB")
or_ok    = _check_key(OPENROUTER_API_KEY, "OpenRouter")

print(f"TMDB API Key:       {'✅ Set' if tmdb_ok  else '❌ MISSING — paste your key above'}")
print(f"OpenRouter Key:     {'✅ Set' if or_ok  else '❌ MISSING — paste your key above'}")

if tmdb_ok:
    try:
        r = requests.get(f"{TMDB_BASE}/configuration", params={"api_key": TMDB_API_KEY}, timeout=5)
        print("TMDB connection:    ✅ Connected" if r.status_code == 200 else f"TMDB connection:   ⚠️  Status {r.status_code}")
    except Exception as e:
        print(f"TMDB connection:    ⚠️  {e}")

TMDB API Key:       ✅ Set
OpenRouter Key:     ✅ Set
TMDB connection:    ✅ Connected


In [4]:
# CELL 3: TMDB Tool Functions (given as tools to the agent)

def _get(endpoint: str, params: dict = None) -> dict:
    p = {"api_key": TMDB_API_KEY, **(params or {})}
    try:
        r = requests.get(f"{TMDB_BASE}{endpoint}", params=p, timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"error": str(e), "results": []}

def _fmt_movie(m: dict) -> dict:
    return {
        "id":           m.get("id"),
        "title":        m.get("title", "Unknown"),
        "year":         (m.get("release_date") or "")[:4],
        "rating":       round(m.get("vote_average", 0), 1),
        "overview":     (m.get("overview") or "")[:220],
        "poster_path":  m.get("poster_path"),
    }

def search_movies(query: str, page: int = 1) -> dict:
    data = _get("/search/movie", {"query": query, "page": page, "include_adult": "false"})
    return {"query": query, "movies": [_fmt_movie(m) for m in data.get("results", [])[:8]]}

def get_movie_details(movie_id: int) -> dict:
    data = _get(f"/movie/{movie_id}", {"append_to_response": "credits,videos,similar"})
    if "status_code" in data: return {"error": "Not found"}

    director = next((c["name"] for c in data.get("credits", {}).get("crew", []) if c["job"] == "Director"), "")
    cast = [{"name": a["name"], "character": a["character"]} for a in data.get("credits", {}).get("cast", [])[:5]]
    trailer_key = next((v["key"] for v in data.get("videos", {}).get("results", []) if v["type"] == "Trailer" and v["site"] == "YouTube"), None)

    return {
        "id": data["id"], "title": data.get("title"),
        "year": (data.get("release_date") or "")[:4],
        "rating": round(data.get("vote_average", 0), 1),
        "runtime": data.get("runtime"),
        "overview": data.get("overview", ""),
        "genres": [g["name"] for g in data.get("genres", [])],
        "poster_path": data.get("poster_path"),
        "director": director, "cast": cast,
        "trailer_url": f"https://youtu.be/{trailer_key}" if trailer_key else None,
        "similar_movies": [_fmt_movie(m) for m in data.get("similar", {}).get("results", [])[:5]],
    }

def discover_movies_by_mood(mood: str, sort_by: str = "vote_average.desc", page: int = 1) -> dict:
    genre_ids = MOOD_GENRE_MAP.get(mood.lower(), MOOD_GENRE_MAP["neutral"])
    genre_str = ",".join(map(str, genre_ids[:2]))
    data = _get("/discover/movie", {
        "with_genres": genre_str, "sort_by": sort_by,
        "vote_count.gte": 300, "vote_average.gte": 6.5, "page": page
    })
    return {"mood": mood, "movies": [_fmt_movie(m) for m in data.get("results", [])[:8]]}

def get_trending_movies(time_window: str = "week") -> dict:
    data = _get(f"/trending/movie/{time_window}")
    return {"time_window": time_window, "movies": [_fmt_movie(m) for m in data.get("results", [])[:8]]}

def get_now_playing() -> dict:
    data = _get("/movie/now_playing", {"region": "US"})
    return {"movies": [_fmt_movie(m) for m in data.get("results", [])[:8]]}

def get_top_rated_movies(genre: str = None) -> dict:
    if genre:
        gid = GENRE_IDS.get(genre.lower())
        data = _get("/discover/movie", {"with_genres": gid, "sort_by": "vote_average.desc", "vote_count.gte": 500}) if gid else _get("/movie/top_rated")
    else:
        data = _get("/movie/top_rated")
    return {"genre_filter": genre, "movies": [_fmt_movie(m) for m in data.get("results", [])[:8]]}

_watchlist: dict = {}
def manage_watchlist(action: str, user_id: str = "default", movie_id: int = None, movie_title: str = None, poster_path: str = None, rating: float = None) -> dict:
    _watchlist.setdefault(user_id, {"to_watch": [], "watched": []})
    wl = _watchlist[user_id]
    if action == "add":
        if not any(m["id"] == movie_id for m in wl["to_watch"]):
            wl["to_watch"].append({"id": movie_id, "title": movie_title, "poster_path": poster_path, "rating": rating})
            return {"success": True, "message": f"✅ '{movie_title}' added!"}
        return {"success": False, "message": "Already in watchlist."}
    elif action == "mark_watched":
        entry = next((m for m in wl["to_watch"] if m["id"] == movie_id), None)
        if entry:
            wl["to_watch"].remove(entry)
            wl["watched"].append(entry)
            return {"success": True, "message": f"🎉 Marked '{entry['title']}' as watched!"}
        return {"success": False, "message": "Not found in to-watch."}
    elif action == "view":
        return {"to_watch": wl["to_watch"], "watched": wl["watched"], "total_to_watch": len(wl["to_watch"]), "total_watched": len(wl["watched"])}
    return {"error": "Unknown action"}

print("✅ Cell 3 ready — TMDB tools defined")

✅ Cell 3 ready — TMDB tools defined


In [5]:
# CELL 4: Tool Schemas for the Agent

TOOLS = [
    {
        "name": "search_movies",
        "description": "Search for movies on TMDB by title, actor name, director, or keyword.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}, "page": {"type": "integer", "default": 1}},
            "required": ["query"]
        }
    },
    {
        "name": "get_movie_details",
        "description": "Fetch FULL details for one movie using its TMDB ID.",
        "input_schema": {
            "type": "object",
            "properties": {"movie_id": {"type": "integer"}},
            "required": ["movie_id"]
        }
    },
    {
        "name": "discover_movies_by_mood",
        "description": "★ PRIMARY TOOL ★ — Discover TMDB movies matched to user's emotional state/mood.",
        "input_schema": {
            "type": "object",
            "properties": {
                "mood": {"type": "string", "description": "One of: joy, happiness, sadness, grief, anger, fear, anxiety, stressed, love, romantic, surprise, excited, bored, nostalgic, curious, adventurous, relaxed, neutral"},
                "sort_by": {"type": "string", "default": "vote_average.desc"},
                "page": {"type": "integer", "default": 1}
            },
            "required": ["mood"]
        }
    },
    {
        "name": "get_trending_movies",
        "description": "Get what's trending on TMDB right now ('day' or 'week').",
        "input_schema": {
            "type": "object",
            "properties": {"time_window": {"type": "string", "default": "week"}},
            "required": []
        }
    },
    {
        "name": "get_now_playing",
        "description": "Get movies CURRENTLY in cinemas.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "get_top_rated_movies",
        "description": "Get the all-time highest-rated movies. Optionally filter by genre.",
        "input_schema": {
            "type": "object",
            "properties": {"genre": {"type": "string"}},
            "required": []
        }
    },
    {
        "name": "manage_watchlist",
        "description": "Add movies, mark watched, or view the user's watchlist.",
        "input_schema": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "description": "'add', 'mark_watched', 'view'"},
                "movie_id": {"type": "integer"}, "movie_title": {"type": "string"},
                "poster_path": {"type": "string"}, "rating": {"type": "number"}
            },
            "required": ["action"]
        }
    }
]
print("✅ Cell 4 ready — Tool schemas registered")

✅ Cell 4 ready — Tool schemas registered


In [6]:
# CELL 5: Tool Executor

TOOL_REGISTRY = {
    "search_movies": search_movies, "get_movie_details": get_movie_details,
    "discover_movies_by_mood": discover_movies_by_mood, "get_trending_movies": get_trending_movies,
    "get_now_playing": get_now_playing, "get_top_rated_movies": get_top_rated_movies,
    "manage_watchlist": manage_watchlist,
}

def execute_tool(name: str, inputs: dict) -> str:
    fn = TOOL_REGISTRY.get(name)
    if not fn: return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return json.dumps(fn(**inputs), ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": f"{name} failed: {str(e)}"})

def extract_movies(result: dict, tool_name: str) -> list:
    if tool_name == "get_movie_details": return [result] if result.get("id") else []
    if tool_name == "manage_watchlist": return result.get("to_watch", []) + result.get("watched", [])
    return result.get("movies", [])

def label_for_tool(tool_name: str, inputs: dict) -> str:
    labels = {
        "discover_movies_by_mood": lambda i: f"🎭 Movies for your {i.get('mood','').title()} mood",
        "search_movies": lambda i: f"🔍 Results for '{i.get('query','')}'",
        "get_trending_movies": lambda i: f"🔥 Trending This {i.get('time_window','Week').title()}",
        "get_now_playing": lambda _: "🎦 Now Playing in Theaters",
        "get_top_rated_movies": lambda i: f"⭐ Top Rated {(i.get('genre') or 'Movies').title()}",
        "get_movie_details": lambda _: "🎬 Movie Details",
        "manage_watchlist": lambda _: "📋 Your Watchlist",
    }
    return labels.get(tool_name, lambda _: "🎬 Movies")(inputs)

print("✅ Cell 5 ready — Executor wired up")

✅ Cell 5 ready — Executor wired up


In [7]:
'''# CELL 6: Rich HTML Display (UI)

PLACEHOLDER = "https://placehold.co/160x235/1a1a2e/888888?text=No+Poster"

def movie_card_html(m: dict, detail_mode: bool = False) -> str:
    poster = f"{IMG_W200}{m['poster_path']}" if m.get("poster_path") else PLACEHOLDER
    rating = m.get("rating", 0)
    rc = "#2ecc71" if rating >= 7.5 else "#f39c12" if rating >= 6 else "#e74c3c" if rating > 0 else "#555"
    title = (m.get("title") or "")[:36]

    genre_html = ""
    if m.get("genres"):
        gtags = "".join(f'<span style="background:#1e3a5f;color:#7db3e0;padding:1px 5px;border-radius:3px;font-size:10px;margin:1px;display:inline-block">{g}</span>' for g in m["genres"][:3])
        genre_html = f'<div style="margin-top:4px">{gtags}</div>'

    trailer_btn = (f'<a href="{m["trailer_url"]}" target="_blank" style="display:inline-block;margin-top:6px;padding:3px 9px;background:#c0392b;color:#fff;border-radius:4px;font-size:11px;text-decoration:none">▶ Trailer</a>') if detail_mode and m.get("trailer_url") else ""

    return f"""
<div style="display:inline-block;width:162px;vertical-align:top;margin:5px;background:#111827;border-radius:10px;overflow:hidden;box-shadow:0 6px 20px rgba(0,0,0,.55);">
  <div style="position:relative">
    <img src="{poster}" style="width:162px;height:238px;object-fit:cover;display:block" onerror="this.src='{PLACEHOLDER}'" />
    <div style="position:absolute;top:7px;right:7px;background:{rc};color:#fff;border-radius:999px;padding:2px 8px;font-size:12px;font-weight:700;">{rating if rating else "–"}</div>
  </div>
  <div style="padding:8px 9px 10px">
    <div style="font-size:13px;font-weight:700;color:#f0f0f0;line-height:1.3;min-height:34px">{title}</div>
    <div style="color:#6b7280;font-size:11px;margin-top:1px">{m.get('year','')}</div>
    {genre_html}{trailer_btn}
  </div>
</div>"""

def render_movies(movies: list, section_title: str = "", detail_mode: bool = False):
    if not movies: return
    seen = set()
    unique = [m for m in movies if m.get("id") not in seen and not seen.add(m.get("id"))]
    cards = "".join(movie_card_html(m, detail_mode) for m in unique)
    display(HTML(f"""<div style="background:linear-gradient(180deg,#0f172a,#111827);padding:14px 12px;border-radius:14px;margin:6px 0;border:1px solid #1e293b"><div style="color:#f59e0b;font-size:15px;font-weight:700;margin:12px 0 4px;">{section_title}</div><div style="display:flex;flex-wrap:wrap;gap:2px">{cards}</div></div>"""))

def render_bubble(role: str, text: str):
    bg, align, lbl = ("#1e3a5f", "right", "👤 You") if role == "user" else ("#0f2436", "left", "🎬 CineBot")
    safe = text.replace("**","").replace("\n","<br>")
    display(HTML(f'<div style="margin:5px 0;text-align:{align}"><div style="font-size:11px;color:#6b7280;margin-bottom:2px">{lbl}</div><div style="display:inline-block;max-width:82%;background:{bg};color:#e2e8f0;padding:10px 14px;border-radius:12px;font-size:14px;text-align:left;line-height:1.55;">{safe}</div></div>'))

def show_watchlist_visual(user_id: str = "default"):
    res = manage_watchlist("view", user_id=user_id)
    if res["total_to_watch"] == 0 and res["total_watched"] == 0:
        display(HTML('<p style="color:#6b7280;font-style:italic">Watchlist is empty.</p>'))
        return
    if res["to_watch"]: render_movies(res["to_watch"], f"📋 To Watch ({res['total_to_watch']})")
    if res["watched"]: render_movies(res["watched"], f"✅ Watched ({res['total_watched']})")

print("✅ Cell 6 ready — UI loaded")'''

'# CELL 6: Rich HTML Display (UI)\n\nPLACEHOLDER = "https://placehold.co/160x235/1a1a2e/888888?text=No+Poster"\n\ndef movie_card_html(m: dict, detail_mode: bool = False) -> str:\n    poster = f"{IMG_W200}{m[\'poster_path\']}" if m.get("poster_path") else PLACEHOLDER\n    rating = m.get("rating", 0)\n    rc = "#2ecc71" if rating >= 7.5 else "#f39c12" if rating >= 6 else "#e74c3c" if rating > 0 else "#555"\n    title = (m.get("title") or "")[:36]\n\n    genre_html = ""\n    if m.get("genres"):\n        gtags = "".join(f\'<span style="background:#1e3a5f;color:#7db3e0;padding:1px 5px;border-radius:3px;font-size:10px;margin:1px;display:inline-block">{g}</span>\' for g in m["genres"][:3])\n        genre_html = f\'<div style="margin-top:4px">{gtags}</div>\'\n\n    trailer_btn = (f\'<a href="{m["trailer_url"]}" target="_blank" style="display:inline-block;margin-top:6px;padding:3px 9px;background:#c0392b;color:#fff;border-radius:4px;font-size:11px;text-decoration:none">▶ Trailer</a>\') if detai

In [8]:
# CELL 6: Rich HTML Display (UI) - Upgraded with Cast & Trailers

PLACEHOLDER = "https://placehold.co/160x235/1a1a2e/888888?text=No+Poster"

def _get_extra_details(movie_id):
    """Automatically fetch cast and trailer if the movie doesn't have it yet."""
    try:
        url = f"{TMDB_BASE}/movie/{movie_id}?api_key={TMDB_API_KEY}&append_to_response=credits,videos"
        r = requests.get(url, timeout=3)
        if r.status_code != 200: return [], None
        data = r.json()
        cast = [{"name": a["name"]} for a in data.get("credits", {}).get("cast", [])[:3]]
        trailer_key = next((v["key"] for v in data.get("videos", {}).get("results", []) if v["type"] == "Trailer" and v["site"] == "YouTube"), None)
        return cast, f"https://youtu.be/{trailer_key}" if trailer_key else None
    except:
        return [], None

def movie_card_html(m: dict, detail_mode: bool = False) -> str:
    poster = f"{IMG_W200}{m['poster_path']}" if m.get("poster_path") else PLACEHOLDER
    rating = m.get("rating", 0)
    rc = "#2ecc71" if rating >= 7.5 else "#f39c12" if rating >= 6 else "#e74c3c" if rating > 0 else "#555"
    title = (m.get("title") or "")[:36]

    # 1. Genres
    genre_html = ""
    if m.get("genres"):
        gtags = "".join(f'<span style="background:#1e3a5f;color:#7db3e0;padding:1px 5px;border-radius:3px;font-size:10px;margin:1px;display:inline-block">{g}</span>' for g in m["genres"][:3])
        genre_html = f'<div style="margin-top:4px">{gtags}</div>'

    # 2. Fetch missing Cast & Trailer
    cast = m.get("cast")
    trailer_url = m.get("trailer_url")
    if cast is None or trailer_url is None:
        fetched_cast, fetched_trailer = _get_extra_details(m["id"])
        cast = cast or fetched_cast
        trailer_url = trailer_url or fetched_trailer

    # 3. Format Cast text
    cast_html = ""
    if cast:
        cast_names = ", ".join(c["name"] for c in cast[:3])
        cast_html = f'<div style="color:#9ca3af;font-size:10px;margin-top:8px;line-height:1.3;">🎭 {cast_names}</div>'

    # 4. Format Trailer button
    trailer_btn = ""
    if trailer_url:
        trailer_btn = f'<a href="{trailer_url}" target="_blank" style="display:inline-block;margin-top:8px;padding:5px 12px;background:#e50914;color:#fff;border-radius:4px;font-size:11px;font-weight:bold;text-decoration:none;box-shadow:0 2px 4px rgba(0,0,0,0.4);transition: background 0.2s;">▶ Watch Trailer</a>'

    return f"""
<div style="display:inline-block;width:162px;vertical-align:top;margin:5px;background:#111827;border-radius:10px;overflow:hidden;box-shadow:0 6px 20px rgba(0,0,0,.55);">
  <div style="position:relative">
    <img src="{poster}" style="width:162px;height:238px;object-fit:cover;display:block" onerror="this.src='{PLACEHOLDER}'" />
    <div style="position:absolute;top:7px;right:7px;background:{rc};color:#fff;border-radius:999px;padding:2px 8px;font-size:12px;font-weight:700;">{rating if rating else "–"}</div>
  </div>
  <div style="padding:10px">
    <div style="font-size:13px;font-weight:700;color:#f0f0f0;line-height:1.3;min-height:34px">{title}</div>
    <div style="color:#6b7280;font-size:11px;margin-top:2px">{m.get('year','')}</div>
    {genre_html}
    {cast_html}
    {trailer_btn}
  </div>
</div>"""

def render_movies(movies: list, section_title: str = "", detail_mode: bool = False):
    if not movies: return
    seen = set()
    unique = [m for m in movies if m.get("id") not in seen and not seen.add(m.get("id"))]
    cards = "".join(movie_card_html(m, detail_mode) for m in unique)
    display(HTML(f"""<div style="background:linear-gradient(180deg,#0f172a,#111827);padding:14px 12px;border-radius:14px;margin:6px 0;border:1px solid #1e293b"><div style="color:#f59e0b;font-size:15px;font-weight:700;margin:12px 0 4px;">{section_title}</div><div style="display:flex;flex-wrap:wrap;gap:2px">{cards}</div></div>"""))

def render_bubble(role: str, text: str):
    bg, align, lbl = ("#1e3a5f", "right", "👤 You") if role == "user" else ("#0f2436", "left", "🎬 CineBot")
    safe = text.replace("**","").replace("\n","<br>")
    display(HTML(f'<div style="margin:5px 0;text-align:{align}"><div style="font-size:11px;color:#6b7280;margin-bottom:2px">{lbl}</div><div style="display:inline-block;max-width:82%;background:{bg};color:#e2e8f0;padding:10px 14px;border-radius:12px;font-size:14px;text-align:left;line-height:1.55;">{safe}</div></div>'))

def show_watchlist_visual(user_id: str = "default"):
    res = manage_watchlist("view", user_id=user_id)
    if res["total_to_watch"] == 0 and res["total_watched"] == 0:
        display(HTML('<p style="color:#6b7280;font-style:italic">Watchlist is empty.</p>'))
        return
    if res["to_watch"]: render_movies(res["to_watch"], f"📋 To Watch ({res['total_to_watch']})")
    if res["watched"]: render_movies(res["watched"], f"✅ Watched ({res['total_watched']})")

print("✅ Cell 6 ready — UI loaded with Cast and Trailer support!")

✅ Cell 6 ready — UI loaded with Cast and Trailer support!


In [9]:
# CELL 7: System Prompt

SYSTEM_PROMPT = """
You are **CineBot**, a world-class AI movie companion powered by real-time TMDB data.
You combine the knowledge of a seasoned film critic with the warmth of a friend who
genuinely loves helping people find their perfect movie.

PERSONALITY & TONE
- Enthusiastic, knowledgeable, and warm. Use 1-2 emojis.
- Concise: keep ALL prose responses to 2–4 sentences max. The visual cards carry the detail.
- Acknowledge the user's emotional state briefly before jumping to recommendations.

TOOL SELECTION LOGIC (ALWAYS follow this order)
1. USER DESCRIBES FEELINGS / MOOD  →  discover_movies_by_mood
   Infer mood from context (rough day = sadness/stressed, gym = excited, can't sleep = anxiety/fear).
2. USER NAMES A SPECIFIC MOVIE  →  search_movies, then get_movie_details
3. USER WANTS SIMILAR MOVIES   →  get_movie_details on the source movie
4. "TRENDING / POPULAR"        →  get_trending_movies
5. "IN THEATERS / CINEMA"      →  get_now_playing
6. WATCHLIST COMMANDS          →  manage_watchlist

RESPONSE STRUCTURE (STRICT)
a) Acknowledge the user's state.
b) State the genre/mood you picked.
c) Call out ONE standout pick from the results.
d) ONE natural follow-up question.
DO NOT list out the movies in your text response. The UI will render the movie cards automatically.

WATCHLIST PROTOCOL
When user says "save X" -> search_movies -> manage_watchlist(action="add")
When user says "watched X" -> manage_watchlist(action="mark_watched")
"""
print("✅ Cell 7 ready — System prompt loaded")

✅ Cell 7 ready — System prompt loaded


In [10]:
# CELL 8: Agentic Chat Loop (Using OpenRouter Native API)
import json, requests
from IPython.display import HTML, display, clear_output

_history = []
_user_id = "default"

# 1. Dynamically convert your Anthropic TOOLS (from Cell 4) into OpenRouter/OpenAI format
OPENAI_TOOLS = []
for t in TOOLS:
    OPENAI_TOOLS.append({
        "type": "function",
        "function": {
            "name": t["name"],
            "description": t["description"],
            "parameters": t["input_schema"]
        }
    })

def chat(message: str, user_id: str = "default"):
    global _history, _user_id
    _user_id = user_id

    render_bubble("user", message)
    _history.append({"role": "user", "content": message})

    all_movies = []
    section_ttl = ""
    detail_mode = False
    watchlist_op = False

    while True:
        try:
            # 2. Build the payload for OpenRouter using the correct model name
            payload = {
                "model": "nvidia/nemotron-3-super-120b-a12b:free",
                "messages": [{"role": "system", "content": SYSTEM_PROMPT}] + _history,
                "tools": OPENAI_TOOLS
            }

            headers = {
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "HTTP-Referer": "https://colab.research.google.com/",
                "X-Title": "CineBot",
                "Content-Type": "application/json",
            }

            # 3. Hit OpenRouter's reliable chat/completions endpoint
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=30
            )

            if resp.status_code != 200:
                raise Exception(f"API Error {resp.status_code}: {resp.text[:250]}")

            data = resp.json()
            message_obj = data["choices"][0]["message"]
            msg_dict = {"role": "assistant"}

            if message_obj.get("content"):
                msg_dict["content"] = message_obj["content"]
                render_bubble("assistant", message_obj["content"])

            tool_calls = message_obj.get("tool_calls", [])
            if tool_calls:
                msg_dict["tool_calls"] = tool_calls

            _history.append(msg_dict)

            if not tool_calls:
                break

            # 4. Handle Tool Calls
            for tool_call in tool_calls:
                fn_name = tool_call["function"]["name"]
                args_raw = tool_call["function"]["arguments"]
                args = json.loads(args_raw) if args_raw else {}

                raw_result = execute_tool(fn_name, args)
                result_dict = json.loads(raw_result)

                movies = extract_movies(result_dict, fn_name)
                if movies:
                    all_movies.extend(movies)
                    if fn_name != "manage_watchlist":
                        section_ttl = label_for_tool(fn_name, args)
                    if fn_name == "get_movie_details": detail_mode = True
                    if fn_name == "manage_watchlist": watchlist_op = True

                _history.append({
                    "role": "tool",
                    "tool_call_id": tool_call["id"],
                    "name": fn_name,
                    "content": raw_result,
                })

        except Exception as e:
            display(HTML(f'<div style="color:#ef4444;background:#1e1b4b;padding:10px;border-radius:8px;border:1px solid #ef4444"><b>Connection Error:</b> {str(e)}</div>'))
            break

    # 5. Render Final UI
    if all_movies:
        render_movies(all_movies, section_ttl, detail_mode=detail_mode)
    elif watchlist_op and not all_movies:
        show_watchlist_visual(_user_id)

def reset():
    global _history
    _history = []
    clear_output()
    display(HTML("""
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 100%);padding:24px 20px;border-radius:16px;text-align:center;border:1px solid #312e81;margin:10px 0">
  <div style="font-size:36px;margin-bottom:6px">🎬</div>
  <div style="color:#f1f5f9;font-size:22px;font-weight:800;letter-spacing:-.3px">CineBot (OpenRouter Edition)</div>
  <div style="color:#818cf8;font-size:13px;margin-top:3px">AI Movie Assistant · TMDB + Claude 3.5</div>
</div>"""))

print("✅ Cell 8 ready — Chat Engine Loaded!")

✅ Cell 8 ready — Chat Engine Loaded!


In [11]:
# CELL 9: Start the UI
reset()

In [20]:
# CELL 10: Talk to your Bot! (Run this cell multiple times with different messages)

chat("What's playing in theaters right now?")

# --- OTHER COMMANDS TO TRY (Uncomment one and run!) ---
# chat("It's date night and I want something romantic")
# chat("What's playing in theaters right now?")
# chat("Tell me everything about Interstellar")
# chat("Save Inception to my watchlist")
# show_watchlist_visual()